# Agon: Eos, training intelligent agents (full notebook, current code)

One goal: agents whose decisions you can look at and call intelligent, and a number that says so.

**Everything in here is new code.** Old checkpoints cannot be resumed (the Genotype and the networks changed, `experiment.py` refuses to mix code versions). Each run goes in a new folder `intel_<code version>_<preset>_w<worlds>`.

**What changed since the last notebook, and where it is tested**

| change | where you see it |
|---|---|
| Animals have their own temper; a tame one is neutral to strangers and provoked by a blow to it or its person; anger aims at whoever struck (or, if unseen, whom the animal trusts least) | step 4 tests; step 10 stories |
| World model: ideas standardised over the candidates, own-state head learns the *change* a tick brings | step 8 plots: trust, idea errors |
| `decision_check`: only live threats count; new rules `get_away_from_predator` and WASTE rows (drinking or eating when full) | steps 6 and 8 tables |
| **Entropy floor** (`--entropy-target`) and **no retention** (`--retention-coef 0`): the stall ablations | step 7b |
| Tuning designs: a higher floor (0.8), floor + no retention | step 7c |
| Shared trunk (`--share-trunk`): kept as a flag, **not** run here (you dislike it) | n/a |
| Cold deaths split open / sheltered / adult / child | step 8 stall panel |
| The **ledger**: every chance the world rolls, by source, with its factors and outcome (infection, falls, predator attacks, conception, weather spells, lightning) | step 10 stories: `Odds: P(x) = p% from ... : outcome` |
| The **minds log**: every awake agent's decision every tick, with options, probabilities, what it leaned on, its ideas | step 10 `--minds` |

**Order.** 1 GPU, 2 code, 3 Drive, 4 tests, 5 benchmark, 6 the bar (random and untrained), 7 train the baseline, 7b the two stall ablations, 7c tuning if 7b says so, 8 compare, 9 look at decisions, 10 stories and the ledger.

**What "intelligent" means here, and what it does not.** `decision_check.py` reads each awake agent's choice every tick and asks whether it matches a plain survival rule (drink or seek water when thirsty, eat or seek food when hungry, sleep or rest at night, flee or get away when a predator is within 3 cells), and counts WASTE (drinking or eating when full, where lower is better), then divides by what random agents do. Beating chance on all of them, while living longer than untrained agents, is the minimum. It does not prove planning or culture: the playback, the stories and the Ideas tab are where you look for that.

**Time.** Not measured on this code. The first run on the old code took about 35 h for four 100M-step runs on an A100 (about 9 h each). Step 5 benchmarks the real speed. Each design in step 7 is one such run, so 7, 7b and 7c are 9 h each; do them in order and stop when the answer is clear.

## 1. GPU check

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
!nvidia-smi --query-gpu=name,memory.total --format=csv
import jax
print("JAX", jax.__version__, "devices:", jax.devices())
assert jax.default_backend() == "gpu", "No GPU: Runtime > Change runtime type > A100 GPU, then restart."

## 2. Load the code

Upload the new `lifesim_code.zip` (made from the commit you are on: `git archive --format=zip -o lifesim_code.zip HEAD`), or set `ZIP_FROM_DRIVE`. The cell checks that the zip has the new pieces (ledger, entropy floor, tuning designs) and stops if it does not.

In [ ]:
import glob, html, json, shutil, subprocess, sys, zipfile
ZIP_FROM_DRIVE = ""   # e.g. "/content/drive/MyDrive/agon/lifesim_code.zip"
if ZIP_FROM_DRIVE and os.path.exists(ZIP_FROM_DRIVE):
    zip_path = ZIP_FROM_DRIVE
else:
    from google.colab import files
    zip_path = next(iter(files.upload()))
shutil.rmtree("/content/lifesim", ignore_errors=True)
os.makedirs("/content/lifesim")
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content/lifesim")
if not os.path.exists("/content/lifesim/pyproject.toml"):
    inner = next(os.path.dirname(p) for p in glob.glob("/content/lifesim/*/pyproject.toml"))
    for name in os.listdir(inner):
        shutil.move(os.path.join(inner, name), "/content/lifesim")
os.chdir("/content/lifesim")
!pip install -q -U "flax>=0.12.10" "optax>=0.2.8" pytest
!pip install -q --no-deps -e .
import importlib.metadata as md
v = {p: md.version(p) for p in ("jax", "flax", "optax")}
print(v)
assert tuple(int(x) for x in v["flax"].split(".")[:3]) >= (0, 12, 10), "Flax too old: Runtime > Restart session, rerun this cell."
sys.path.insert(0, "/content/lifesim")
from lifesim.presets import PRESETS, code_version
from lifesim import env as E
PRESET = "society30"   # 32 x 32 at half-hour ticks: 30x less compute per game day than `society`
assert os.path.exists("scripts/decision_check.py"), "This zip predates decision_check.py: upload the new lifesim_code.zip."
assert os.path.exists("scripts/narrate.py"), "This zip predates narrate.py: upload the new lifesim_code.zip."
assert "odds" in E.StepInfo._fields, "This zip predates the ledger (StepInfo.odds): upload the new lifesim_code.zip."
assert "--entropy-target" in open("scripts/train_phase2.py").read(), "This zip predates the entropy floor."
assert "caveman_floor_no_retention" in open("scripts/experiment.py").read(), "This zip predates the tuning designs."
print("code version", code_version(), "| observation channels", E.N_OBS_CHANNELS, "| preset", PRESET)
assert E.N_OBS_CHANNELS == 64, "Not the current code (expected 64 observation channels)."

## 3. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
BASE = f"/content/drive/MyDrive/agon/runs/intel_{code_version()}_{PRESET}"
os.makedirs(BASE, exist_ok=True)
print("saved to", BASE)

## 4. Tests and viability

Two groups. The **new-systems** tests (taming and the anger target, corpses, room, trees, the entropy floor, the shared-trunk flag, the world model, the causal recorder and ledger) are what this notebook depends on; most of the time is compiling. I measured only the causal tests (about 40 s on CPU); the rest I have not timed on Colab, so the cell prints each test as it finishes. If one fails, stop: a failing world is not worth training in.

The rest of the suite (`test_fauna`, `test_living`, `test_env`, ...) takes 30+ minutes and is optional (CI runs it). Then the viability check: can a scripted band live in this world at all?

In [ ]:
!python -m pytest -v -x -p no:cacheprovider tests/test_taming.py tests/test_dead.py tests/test_room.py tests/test_trees.py tests/test_entropy_floor.py tests/test_shared_trunk.py tests/test_world_model.py tests/test_causal.py

In [ ]:
# Optional, 30+ minutes: the rest of the suite. Skip it to start training.
RUN_FULL_SUITE = False
if RUN_FULL_SUITE:
    !python -m pytest -v -x -p no:cacheprovider tests/test_fauna.py tests/test_living.py tests/test_env.py tests/test_features.py

In [ ]:
!python scripts/viability.py --preset {PRESET} --knowledge own --days 160 --seeds 3

## 5. Benchmark

How many parallel worlds fit on this GPU, and how fast. The full update ran out of memory at too many worlds on the earlier code; this picks the largest that fits. `ROOT` has the world count in its name so a checkpoint is never resumed with another batch size.

In [ ]:
WORLDS = None   # set 4 or 8 to skip the benchmark
if WORLDS is None:
    out = subprocess.run([sys.executable, "scripts/benchmark.py", "--preset", PRESET, "--worlds", "4", "8",
                          "--updates", "3", "--target-agent-steps", "1e8"], capture_output=True, text=True)
    print(out.stdout[-2500:] or out.stderr[-2500:])
    rec = [l for l in out.stdout.splitlines() if l.startswith("RECOMMENDED")]
    WORLDS = int(rec[-1].split("--worlds")[1].split()[0]) if rec else 4
ROOT = f"{BASE}_w{WORLDS}"      # one folder per world count: a checkpoint is never resumed with another batch size
os.makedirs(ROOT, exist_ok=True)
print("WORLDS =", WORLDS, "| runs in", ROOT)

## 6. The bar: random and untrained agents

Three seeds of 10 days. Random agents show chance; untrained networks show what a fresh brain does. The table has the survival rules (higher = better), the WASTE rows (acting on a full need; lower = better) and the share alive at the end. This is what training must clear.

In [ ]:
!python scripts/decision_check.py --days 10 --seeds 3 --out {ROOT}/decision_check_untrained.json

## 7. Train the baseline

The `caveman` design (`--only 0`): given survival skills, a world model, sleep-consolidated learning, survival as the only reward, no entropy floor. Then a 400-day continuous world with playback pages and the minds log (about 4 KB per agent-day, so tens of MB for the 400 days). It resumes from its checkpoint: after a disconnect, rerun steps 1, 2, 3 and 5 (set `WORLDS` by hand to skip the benchmark), then this cell.

The first run's log (`claude/research/design-audit-learning.md`) is the reference this must be compared with, but that run was on the old code, so the baseline here is the fair comparison for 7b and 7c.

In [ ]:
STEPS = 1e8
!python scripts/experiment.py validate --root {ROOT} --preset {PRESET} --worlds {WORLDS} --agent-steps {STEPS} --playback-every 4 --only 0

## 7b. Why they stop improving: the stall ablations

On the old code, deaths per sixth of training went 1,025 → 325 → 280 → 177 → 209 → 276, and policy entropy fell to 0.07 nats (of 4.4 possible) a sixth of the way in and stayed there. The agents learned to drink, eat and sleep early, then became near-deterministic and stopped getting better at the rest (huddling on winter nights rose 30% → 64% and stuck there; cold deaths fell 271 → 49 per sixth, then rose; fire never). Two things hold them there: a small fixed entropy bonus, and the retention loss that pulls each brain toward what it already does.

* `--only 1` is `caveman_entropy_floor` (`--entropy-target 0.4`): the entropy bonus grows when entropy is under 0.4 nats and shrinks above it.
* `--only 2` is `caveman_no_retention` (`--retention-coef 0`).

Same preset, steps and seed as step 7. Run one cell, then the other. Each is a separate 9 h run; do not run them at the same time on one GPU.

In [ ]:
ABLATION = 1    # 1 entropy floor, 2 no retention (run the cell twice, once with each)
!python scripts/experiment.py validate --root {ROOT} --preset {PRESET} --worlds {WORLDS} --agent-steps {STEPS} --playback-every 4 --only {ABLATION}

## 7c. Tuning (only after step 8 has compared 0, 1 and 2)

Read the comparison first. Then:

* the 0.4 floor helped (deaths kept falling past the first sixth, huddling rose past 64%) but entropy sat at the floor and learning flattened again → `--only 7`, `caveman_entropy_floor_high` (target 0.8: a stronger push to keep exploring);
* both the floor and no retention helped on their own → `--only 8`, `caveman_floor_no_retention`, both together;
* neither helped → the stall is not in the optimiser; do not spend more runs on it. The next suspect is planning (cold deaths in the open), in `design-audit-learning.md`.

Same preset and steps. This cell trains one of them.

In [ ]:
TUNING = 7      # 7 floor target 0.8, 8 floor 0.4 + no retention
!python scripts/experiment.py validate --root {ROOT} --preset {PRESET} --worlds {WORLDS} --agent-steps {STEPS} --playback-every 4 --only {TUNING}

## 8. Are they getting intelligent? Compare every run

Do not run these while a training cell is still going in another session: two heavy jobs on one GPU run out of memory. It reads the latest checkpoint if training has not finished.

**Ideas.** The world model gives each agent imagined futures for a few candidate acts (Ideas tab). The agent acts on them by a learned weight, `trust`, that starts at 0: ideas change nothing until acting on them pays. In the plots below, trust staying at 0 means the ideas are decoration; trust above 0 with falling idea errors means they are being used; negative means they mislead. `decision_check.py` does not score ideas.

In [ ]:
# Which runs exist so far
NAMES = [n for n in ("caveman", "caveman_entropy_floor", "caveman_no_retention",
                     "caveman_entropy_floor_high", "caveman_floor_no_retention")
         if os.path.exists(f"{ROOT}/validate/{n}/history.json")]
print(NAMES)

In [ ]:
# One decision_check per run (10 days x 3 seeds, each run against random and untrained). Skips those already scored.
for n in NAMES:
    out = f"{ROOT}/decision_check_{n}.json"
    if os.path.exists(out): print("have", out); continue
    !python scripts/decision_check.py --run {ROOT}/validate/{n} --days 10 --seeds 3 --out {out}

In [ ]:
# Side by side: rate per rule for random, untrained and each trained run; WASTE rows (lower = better); alive at end
tabs = {n: json.load(open(f"{ROOT}/decision_check_{n}.json"))["table"] for n in NAMES}
first = tabs[NAMES[0]]
rules = [r for r in first["random"] if r != "alive_share_end"]
fmt = lambda v: "n/a" if v is None else f"{v:.1%}"
print(f"{'rule':<26}{'random':>9}{'untrained':>11}" + "".join(f"{n.replace('caveman', '', 1).strip('_') or 'base':>20}" for n in NAMES))
for r in rules:
    print(f"{r:<26}{fmt(first['random'][r]['rate']):>9}{fmt(first['untrained'][r]['rate']):>11}"
          + "".join(f"{fmt(tabs[n]['trained'][r]['rate']):>20}" for n in NAMES))
print(f"{'alive at end':<26}{first['random']['alive_share_end']:>9.0%}{first['untrained']['alive_share_end']:>11.0%}"
      + "".join(f"{tabs[n]['trained']['alive_share_end']:>20.0%}" for n in NAMES))
print("WASTE rows (drink_when_full, eat_when_full): lower is better. Others: higher is better.")

In [ ]:
# The stall panel for every run: is learning still moving? Deaths by cause per sixth of training (fewer = better; read
# with births, since more mouths means more deaths), mean age at death, policy entropy, winter-night huddling.
import collections
def sixths(run):
    hist = json.load(open(f"{run}/history.json"))["history"]
    K = 6; sz = max(1, len(hist) // K); rows = []
    for i in range(K):
        seg = hist[i * sz:(i + 1) * sz] if i < K - 1 else hist[i * sz:]
        if not seg: continue
        d = collections.Counter(); cold = collections.Counter()
        for r in seg: d.update(r["deaths"]); cold.update(r.get("cold_deaths", {}))
        ages = [r["mean_age_at_death"] for r in seg if r.get("mean_age_at_death") is not None]
        hud = [r["winter_huddle_share"] for r in seg if r.get("winter_huddle_share") is not None]
        rows.append((i, sum(d.values()), dict(d.most_common(4)), sum(r["births"] for r in seg),
                     sum(ages) / max(1, len(ages)), sum(r["entropy"] for r in seg) / len(seg),
                     sum(hud) / len(hud) if hud else float("nan"), dict(cold)))
    return rows
for n in NAMES:
    print("\n==", n)
    print(f"{'sixth':>5} {'deaths':>6} {'births':>6} {'age at death':>13} {'entropy':>8} {'huddle':>7}  top causes")
    for i, k, top, b, age, ent, hud, cold in sixths(f"{ROOT}/validate/{n}"):
        print(f"{i:>5} {k:>6} {b:>6} {age:>13.0f} {ent:>8.3f} {hud:>7.2f}  {top}")
        if cold: print(f"{'':>5} cold: {cold}")
print("\nStalled if deaths stop falling after the first sixth while entropy sits under ~0.1.")
print("Cold: 'open' deaths are a planning failure (a winter night in the open is damage, huddled or not);")
print("'sheltered' ones mean snow, a cold snap, or not resting; 'child' ones are the infants' own.")

In [ ]:
# Curves for all runs on the same axes
import matplotlib.pyplot as plt
H = {n: json.load(open(f"{ROOT}/validate/{n}/history.json"))["history"] for n in NAMES}
share = lambda r, k: r["deaths"].get(k, 0) / max(1, sum(r["deaths"].values()))
panels = [("Mean age at death (ticks)", lambda r: r["mean_age_at_death"]),
          ("Thirst's share of deaths", lambda r: share(r, "thirst")),
          ("Starvation's share of deaths", lambda r: share(r, "starvation")),
          ("Winter-night huddling", lambda r: r.get("winter_huddle_share")),
          ("Asleep at night", lambda r: r.get("asleep_share_night")),
          ("Policy entropy (nats)", lambda r: r.get("entropy")),
          ("Births per window", lambda r: r.get("births")),
          ("Share of experience learned", lambda r: r.get("learned_share")),
          ("Trust in ideas (0 = ignored)", lambda r: (r.get("world_model") or {}).get("trust_in_ideas")),
          ("Idea error: own body", lambda r: (r.get("world_model") or {}).get("own_state_error")),
          ("Idea error: what others do", lambda r: (r.get("world_model") or {}).get("belief_error")),
          ("Idea error: reward", lambda r: (r.get("world_model") or {}).get("reward_error"))]
fig, axes = plt.subplots(3, 4, figsize=(20, 11))
for ax, (title, f) in zip(axes.flat, panels):
    for n in NAMES:
        ax.plot([r["agent_steps"] / 1e6 for r in H[n]], [float("nan") if f(r) is None else f(r) for r in H[n]], label=n.replace("caveman", "", 1).strip("_") or "baseline", lw=1)
    ax.set_title(title, fontsize=10); ax.set_xlabel("million agent-steps")
axes.flat[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**How to read it.** A change is real only if it shows in the table *and* the stall panel *and* the curves, and holds against the baseline run on this same code. One seed per design: differences of a few percent are noise. Things I will look for when you paste these:

* entropy under the floor design should sit near 0.4, not 0.07; if the policy is just noisy, alive-at-end and the survival rules should *drop*, not rise;
* winter huddling above 64% and cold-open deaths falling would be the first real evidence of learning past the stall;
* trust in ideas above 0 with the own-body error falling is the world model being used;
* WASTE rates well under random mean the agents stopped drinking and eating when full.

## 9. Look at their decisions

A one-day playback of the trained agents. Click a person, then the Inspect tab: Decisions lists the acts it weighed with the policy's own probabilities and what it leaned on; Ideas shows what its world model imagined. Compare with an untrained page made the same way (`--decisions`). `RUN` picks the run: change it to look at another.

In [ ]:
RUN = f"{ROOT}/validate/caveman"      # or caveman_entropy_floor, caveman_no_retention, ...
subprocess.run([sys.executable, "scripts/export_playback.py", "--run", RUN, "--days", "1", "--out", "/content/trained_decisions.html"], check=True)
subprocess.run([sys.executable, "scripts/export_playback.py", "--decisions", "--world-model", "--world", "living", "--slate", "caveman",
                "--agents", "8", "--grid", "32", "--days", "1", "--out", "/content/untrained_decisions.html"], check=True)
from IPython.display import HTML, display
display(HTML(f'<iframe srcdoc="{html.escape(open("/content/trained_decisions.html").read())}" style="width:100%;height:1100px;border:0"></iframe>'))

In [ ]:
# Download both pages to open in your browser
from google.colab import files
files.download("/content/trained_decisions.html"); files.download("/content/untrained_decisions.html")

### 9b. The decisions as text

The playback pages are about 28 MB each. This reads the decisions out of them as plain text: how confident the policy was, what agents actually chose when thirsty, hungry, cold, hurt or near a predator, and a spread of moments with the options weighed (with probabilities), what the choice leaned on, and the imagined plan with its expected effect on the body (Ideas). Saved next to the run, a few KB, so it can be read from Drive. For one agent: add `--agent 3`.

In [ ]:
CHUNKS = sorted(glob.glob(f"{RUN}/continuous/chunk_*.html"))
print([os.path.basename(c) for c in CHUNKS])
!python scripts/show_decisions.py {' '.join(CHUNKS)} --max 8 --out {RUN}/decisions_report.txt

## 10. Cause and effect, the stories, and the ledger

The continuous world (400 days) writes `events.jsonl.gz` (every event with its parents and causes), and `minds.jsonl.gz` (every awake agent's decision every tick). This turns them into text.

* `causal_story.py`: the chains that ended in a death or a birth.
* `narrate.py`: the run as a story, every sentence from the record; nothing invented. Where the world rolled a chance, the line says it: `Odds: P(x) = p% from ... (factors): outcome`. Where the record has no cause it says so ("source not traced"), it does not guess.
* `narrate.py --minds`: adds each agent's body, options with probabilities, what it chose and leaned on, its ideas, then the effect and cause of the same tick. It is large.
* The ledger covers: infection (water, meat, contact, bite, rot, immunity, resistance, drought, marsh, foul, sick neighbours), falls (hazard, injury, night, sleepiness), predator attacks (target, day, temper, wildness, deterrence, defenders), conception (and what blocked it), the weather's chance of an extreme and the season's shares, lightning in a storm. The raw random draw is kept as a field, not printed in the story.

In [ ]:
!python scripts/causal_story.py {RUN}/continuous --max 40
!python scripts/experiment.py report --root {ROOT}
!python scripts/narrate.py {RUN}/continuous | tail -40            # the run as a story, from the record: narrative.txt
!python scripts/narrate.py {RUN}/continuous --minds > /dev/null  # every decision and thought: narrative_minds.txt (large)
!ls -la {RUN}/continuous/narrative*.txt

In [ ]:
# What the ledger says about this run: the odds the story quotes ("fell (a steep edge; P = 2.8% ...)",
# "(the spell ended; P(an extreme) = 40.6% ...)"), grouped by what rolled.
import re, collections
txt = open(f"{RUN}/continuous/narrative.txt").read()
by = collections.defaultdict(list)
for m in re.finditer(r"\(([^()]*?)P(?:\(([^)]*)\))? = ([0-9.]+)%", txt):
    by[(m.group(2) or m.group(1)).strip(" ;:")[:40]].append(float(m.group(3)))
print(sum(len(v) for v in by.values()), "odds quoted in the story")
for k, v in sorted(by.items(), key=lambda kv: -len(kv[1]))[:15]:
    print(f"{k:<42} n={len(v):<5} mean P={sum(v)/len(v):5.1f}%  max P={max(v):5.1f}%")
print("untraced sources:", txt.count("source not traced"))

### 10b. One life, or one stretch

Read a single agent's day-by-day with every thought (set `AGENT`, `FROM`, `TO`), for example the last days of someone who died, or the first winter. Use `--minds` for the decisions.

In [ ]:
AGENT, FROM, TO = 3, 0, 10
!python scripts/narrate.py {RUN}/continuous --minds --agent {AGENT} --from-day {FROM} --to-day {TO} --out {RUN}/continuous/agent_{AGENT}.txt | head -120

## 11. What to paste back

For the next round, paste (or point me at on Drive):

1. step 6's table (untrained) and step 8's side-by-side table;
2. step 8's stall panel for every run;
3. `{ROOT}/validate/<run>/decisions_report.txt` for the baseline and the best ablation;
4. the last 40 lines of step 10's story and the ledger summary.

Everything under `{ROOT}` is on Drive and survives the session.